# Laboratorium 1: Dekoratory, Deskryptory i Generatory
### Skoroszyt główny

---

## Cele Laboratorium
Celem dzisiejszych zajęć jest opanowanie zaawansowanych konstrukcji języka Python, które są niezbędne do projektowania nowoczesnej architektury aplikacji.

### System Wspomagania AI (Tutor)
W trakcie rozwiązywania zadań możesz korzystać z pomocy dedykowanego tutora AI. System oferuje 6 poziomów wsparcia:
1. **Ogólna wskazówka**: Sugestia kierunku rozwiązania.
2. **Pseudokod**: Logiczny opis algorytmu.
3. **Mały fragment kodu**: Kluczowa linia lub konstrukcja.
4. **Częściowa implementacja**: Szkielet kodu do uzupełnienia.
5. **Szczegółowe wyjaśnienie**: Analiza mechanizmu działania.
6. **Pełne rozwiązanie**: Dostępne w sytuacjach ostatecznych.

---

## 1. Dekoratory

### DEMO: Dekorator @timer 
Stwórz dekorator @timer, który będzie mierzył i wyświetlał czas wykonania funkcji.

In [ ]:
import time
import functools

def timer(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start_time = time.perf_counter()
        result = func(*args, **kwargs)
        end_time = time.perf_counter()
        print(f"Czas wykonania {func.__name__}: {end_time - start_time:.4f} s")
        return result
    return wrapper

@timer
def example_task():
    time.sleep(0.5)
    print("Zadanie zakończone.")

example_task()

### Zadanie 1: Liczba elementów listy
Stwórz dekorator, który będzie odpowiedzialny za wyświetlanie liczby elementów listy, jeśli jakakolwiek lista pojawi się w parametrach funkcji dekorowanej. 

**Protip:** użyj isinstance do sprawdzenia czy parametr jest listą. Pamiętaj o zachowaniu metadanych funkcji.

In [ ]:
import functools


def show_list_length(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        for i, arg in enumerate(args):
            if isinstance(arg, list):
                print(f"Found list with {len(arg)} elements - arg #{i}")

        for key, value in kwargs.items():
            if isinstance(value, list):
                print(f"Found list with {len(value)} elements - key = {key}")
        return func(*args, **kwargs)
    return wrapper

# Test:
@show_list_length
def process_data(something, opt=None):
    print(f"Inside base function")


process_data('abc')
print("===***===")
process_data(['abc', 'bcd'])
print("===***===")
process_data(['abc', 'bcd'], opt=22)
print("===***===")
process_data(['abc', 'bcd'], opt=[22, 33, 44, 55, 66])
print("===***===")
process_data(1, opt=[22, 55, 66])

### Zadanie 2: Logowanie do pliku
Stwórz dekorator, który będzie zapisywał w pliku *.log nazwę funkcji dekorowanej, datę oraz długość wykonania. Nazwa pliku będzie podana jako argument dekoratora.

**Protip:** użyj biblioteki datetime. Pamiętaj o tym, żeby dekoratory przyjęły metadanych funkcji dekorującej.

In [ ]:
import functools
from datetime import datetime
import time


def logger(filename):
    def decorator(basefn):
        @functools.wraps(basefn)
        def wrapper(*args, **kwargs):
            start_time = datetime.now()
            result = basefn(*args, **kwargs)
            end_time = datetime.now()
            run_time = end_time - start_time
            with open(f'{filename}.log', 'a') as f:
                f.write(
                    f"function={basefn.__name__}, "
                    f"date={start_time.strftime('%Y-%m-%d %H:%M:%S')}, "
                    f"duration={run_time}\n"
                )
            return result
        return wrapper
    return decorator


@logger(filename='basic')
def basic_function():
    time.sleep(0.2)
    print("some basic function")


@logger(filename='other')
def other_function():
    time.sleep(0.5)
    print("some other function")


basic_function()
other_function()

--- 
## 2. Deskryptory

### DEMO: Walidator e-mail klasy Student
Stwórz deskryptor, który będzie działał jako walidator email klasy Student. Klasa Student zawiera pola imie, nazwisko i email. Deskryptor ten powinien sprawdzać poprawność danych wprowadzanych podczas tworzenia lub modyfikowania instancji Student.

In [ ]:
class EmailValidator:
    def __set_name__(self, owner, name):
        self.name = name

    def __set__(self, instance, value):
        if "@" not in value:
            raise ValueError(f"Błędny format adresu email: {value}")
        instance.__dict__[self.name] = value

class Student:
    email = EmailValidator()
    
    def __init__(self, imie, nazwisko, email):
        self.imie = imie
        self.nazwisko = nazwisko
        self.email = email

try:
    s = Student("Jan", "Kowalski", "jan.kowalski@wsei.edu.pl")
    print(f"Utworzono studenta: {s.email}")
    # s.email = "invalid_at_email" # Powinno rzucić błąd
except ValueError as e:
    print(e)

### Zadanie 3: Rejestrowanie dostępu
Stwórz klasę Uzytkownik. Klasa powinna zawierać atrybuty imie i wiek. Opracuj deskryptor, który będzie rejestrował dostęp do tych atrybutów za pomocą logowania. Deskryptor powinien logować informacje o odczycie (__get__) oraz zapisie (__set__) wartości atrybutu.

In [ ]:
import logging

logging.basicConfig(level=logging.INFO, format="%(message)s")


class AccessLogger:
    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, instance, owner):
        if instance is None:
            return self
        logging.info(f"Getting field: {self.name}")
        return instance.__dict__[self.name]

    def __set__(self, instance, value):
        logging.info(f"Setting field: {self.name} with value: {value}")
        instance.__dict__[self.name] = value

class Uzytkownik:
    name = AccessLogger()
    age = AccessLogger()

    def __init__(self, name, age):
        self.name = name
        self.age = age


user_1 = Uzytkownik('John', 33)
user_2 = Uzytkownik('Paul', 57)
print(user_1.name)
print(user_2.age)

--- 
## 3. Generatory i Iteratory

### DEMO: Generator Fibonacciego
Napisz klasę, która będzie implementowała generator ciągu Fibonacciego za pomocą metod magicznych __iter__() i __next__().

In [ ]:
class FibonacciGenerator:
    def __init__(self, limit):
        self.limit = limit
        self.a, self.b = 0, 1
        self.count = 0

    def __iter__(self):
        return self

    def __next__(self):
        if self.count >= self.limit:
            raise StopIteration
        
        result = self.a
        self.a, self.b = self.b, self.a + self.b
        self.count += 1
        return result

fib = FibonacciGenerator(10)
print(list(fib))

### Zadanie 4: Generator ciągu Collatza
Opracuj generator ciągu Collatza. Dla liczby naturalnej n, jeśli n jest parzyste, dziel przez 2; jeśli n jest nieparzyste, pomnóż przez 3 i dodaj 1, zaczynając od określonej liczby początkowej, aż do osiągnięcia wartości 1.

In [ ]:
def collatz_generator(n):
    while n != 1:
        yield n
        if n % 2 == 0:
            n = int(n / 2)
        else:
            n = 3 * n + 1
    yield n

# Test:
for status in collatz_generator(10):
    print(status)

---

## Zadania do zrobienia w domu

Poniższe zadania stanowią rozszerzenie materiału i są przeznaczone dla osób chcących zgłębić temat zaawansowanych konstrukcji języka Python.

### Zadanie dodatkowe 1: Dekorator z autoryzacją
Stwórz dekorator `@require_role(role)`, który przyjmuje nazwę wymaganej roli jako argument. Dekorator powinien sprawdzać, czy w globalnym słowniku `current_user` klucz `role` jest zgodny z wymaganym. Jeśli nie, rzuć `PermissionError`.

In [ ]:
import functools

current_user = {"username": "admin", "role": "superuser"}


def require_role(role):
    def decorator(basefn):
        @functools.wraps(basefn)
        def wrapper(*args, **kwargs):
            if role != current_user['role']:
                raise PermissionError(basefn.__name__)
            return basefn(*args, **kwargs)
        return wrapper
    return decorator

@require_role('user')
def basic_function():
    print("Inside basic function")


@require_role('superuser')
def other_function():
    print("Inside other function")


other_function()
try:
    basic_function()
except PermissionError as ex:
    print(f"Got Permission Error: {ex}")

### Zadanie dodatkowe 2: Deskryptor z walidacją typu
Stwórz deskryptor `Typed`, który przyjmuje typ danych (np. `int`, `str`) w konstruktorze. Deskryptor powinien upewnić się, że zapisywana wartość jest tego typu. Jeśli nie, rzuć `TypeError`.

In [ ]:
class Typed:
    def __init__(self, expected_type):
        self.expected_type = expected_type

    def __set_name__(self, owner, name):
        self.name = name

    def __set__(self, instance, value):
        if not isinstance(value, self.expected_type):
            raise TypeError(
                f"Incorrect type! expected: {self.expected_type.__name__}, actual: {type(value).__name__}"
            )
        instance.__dict__[self.name] = value


class Basic:
    value = Typed(int)

    def __init__(self, value):
        self.value = value

basic_1 = Basic(1)
print(basic_1.value)
try:
    basic_2 = Basic('abc')
except TypeError as e:
    print(e)

### Zadanie dodatkowe 3: Nieskończony generator liczb pierwszych
Opracuj generator `prime_generator`, który zwraca kolejne liczby pierwsze. Następnie użyj wyrażenia generatorowego, aby stworzyć iterator zwracający tylko te liczby pierwsze, które kończą się cyfrą 7.

In [ ]:
def prime_generator():
    n = 2
    while True:
        is_prime = True

        for i in range(2, int(n ** 0.5) + 1):
            if n % i == 0:
                is_prime = False
                break

        if is_prime:
            yield n

        n += 1

primes_ending_with_7 = (p for p in prime_generator() if p % 10 == 7)

for _ in range(15):
    print(next(primes_ending_with_7))
